In [ ]:
!pip install torch

In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader , TensorDataset 
import numpy as np


In [2]:
# Input (temp, rainfall, humidity)
inputs = np.array([[73, 67, 43], [91, 88, 64], [87, 134, 58], 
                   [102, 43, 37], [69, 96, 70], [73, 67, 43], 
                   [91, 88, 64], [87, 134, 58], [102, 43, 37], 
                   [69, 96, 70], [73, 67, 43], [91, 88, 64], 
                   [87, 134, 58], [102, 43, 37], [69, 96, 70]], 
                  dtype='float32')

# Targets (apples, oranges)
targets = np.array([[56, 70], [81, 101], [119, 133], 
                    [22, 37], [103, 119], [56, 70], 
                    [81, 101], [119, 133], [22, 37], 
                    [103, 119], [56, 70], [81, 101], 
                    [119, 133], [22, 37], [103, 119]], 
                   dtype='float32')

inputs = torch.from_numpy(inputs)
targets = torch.from_numpy(targets)

In [3]:
w = torch.randn(3,2 , requires_grad = True)
b = torch.rand(2 , requires_grad = True )

In [4]:
def model(x):
    return x @ w + b

In [5]:
preds = model(inputs)
print(preds)

tensor([[ 142.7290, -197.7664],
        [ 201.2858, -264.0309],
        [ 282.5750, -301.6603],
        [  69.9627, -194.4687],
        [ 239.1868, -257.4203],
        [ 142.7290, -197.7664],
        [ 201.2858, -264.0309],
        [ 282.5750, -301.6603],
        [  69.9627, -194.4687],
        [ 239.1868, -257.4203],
        [ 142.7290, -197.7664],
        [ 201.2858, -264.0309],
        [ 282.5750, -301.6603],
        [  69.9627, -194.4687],
        [ 239.1868, -257.4203]], grad_fn=<AddBackward0>)


In [6]:
def mse(t1,t2):
    diff = t1 - t2
    return torch.sum(diff*diff)/diff.numel()

In [7]:
loss = mse(targets , preds)
print(loss)

tensor(65874.0547, grad_fn=<DivBackward0>)


In [8]:
loss.backward()

In [ ]:
with torch.no_grad():
    w -= w.grad()*1e-5
    b -= b.grad()*1e-5

In [ ]:
w.grad().zero_()
b.grad().zero_()

In [12]:
train_ds = TensorDataset(inputs ,targets)
print(train_ds[0:4])

(tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.]]), tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.]]))


In [13]:
train_d1 = DataLoader(train_ds , batch_size = 5 , shuffle = True)


In [14]:
for xb, yb in train_ds:
    print(xb)
    print(yb)
    break

tensor([73., 67., 43.])
tensor([56., 70.])


In [15]:
preds = model(inputs)
print(preds)

tensor([[ 142.7290, -197.7664],
        [ 201.2858, -264.0309],
        [ 282.5750, -301.6603],
        [  69.9627, -194.4687],
        [ 239.1868, -257.4203],
        [ 142.7290, -197.7664],
        [ 201.2858, -264.0309],
        [ 282.5750, -301.6603],
        [  69.9627, -194.4687],
        [ 239.1868, -257.4203],
        [ 142.7290, -197.7664],
        [ 201.2858, -264.0309],
        [ 282.5750, -301.6603],
        [  69.9627, -194.4687],
        [ 239.1868, -257.4203]], grad_fn=<AddBackward0>)


In [16]:

loss_fn = F.mse_loss
loss = loss_fn(preds,targets)
print(loss)

tensor(65874.0547, grad_fn=<MseLossBackward0>)


In [17]:
model = nn.Linear(3,2)
weight = model.weight
bias = model.bias
print(list(model.parameters()))

[Parameter containing:
tensor([[-0.2775, -0.0053, -0.0558],
        [ 0.3794,  0.3793,  0.0496]], requires_grad=True), Parameter containing:
tensor([-0.3670, -0.2544], requires_grad=True)]


In [18]:
loss_fn = F.mse_loss
loss = loss_fn(preds,model(inputs))


In [19]:

opt = torch.optim.SGD(model.parameters(),lr= 1e-5)

In [20]:
def fit(num_epoch , model , loss_fn , opt , train_d1):
    for epoch in range(num_epoch):
        for xb , yb in train_d1:
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()

        if (epoch + 1) % 10 ==0:
            print(f"{epoch + 1} -- {loss.item()}")
            

In [21]:
num_epoch = 100
fit(num_epoch,model , loss_fn , opt, train_d1)

10 -- 382.47125244140625
20 -- 30.567302703857422
30 -- 203.0198974609375
40 -- 96.53253936767578
50 -- 194.720703125
60 -- 64.19905090332031
70 -- 51.792091369628906
80 -- 25.790515899658203
90 -- 31.584630966186523
100 -- 29.385494232177734
